`RecallDataset` is the shared representation used by dataset analyses, model
factories, losses, fitters and simulators. It is a dictionary of aligned
arrays. Compatibility depends on their meaning and alignment; HDF5 is an
optional way to store them.

## What is a dataset in memory?

One trial is a study list and its subsequent recall sequence. Every field
has the same leading trial axis. Study columns index presentations; recall
columns index responses. Those two widths need not match.

Selected declaration from [`jaxcmr.typing`](https://github.com/githubpsyche/jaxcmr/blob/main/jaxcmr/typing.py):

```python
class RecallDataset(TypedDict, extra_items=Array):
    subject: Integer[Array, "n_trials 1"]
    listLength: Integer[Array, "n_trials 1"]
    pres_itemnos: Integer[Array, "n_trials num_presented"]
    recalls: Integer[Array, "n_trials num_recalled"]
    pres_itemids: NotRequired[Integer[Array, "n_trials num_presented"]]
    rec_itemids: NotRequired[Integer[Array, "n_trials num_recalled"]]
```

The first four fields are required and integer-valued. `NotRequired` fields
may be omitted unless a particular operation needs them; `extra_items=Array`
permits additional trial-aligned arrays. This `TypedDict` documents the
structure for readers and type checkers. It is not a comprehensive runtime
validator.

| Field | Interpretation |
|---|---|
| `subject` | Participant identifier, not an array index. |
| `listLength` | Number of valid study events, including repeated presentations. |
| `pres_itemnos` | Within-list item codes; repeated presentations retain the same code. |
| `recalls` | One-based study positions identifying recalled items, in response order. |
| `pres_itemids` | Optional identities in the experiment's shared stimulus pool. |

The existing war-photo-recall dataset illustrates these fields. Its 492
trials belong to 123 participants. Loading a file does not change the
representation expected by the rest of the library.

In [1]:
#| code-summary: "Inspect the required fields and one study list"

from jaxcmr.helpers import load_data

data = load_data("data/Dupertuys2026.h5")
for name in ("subject", "listLength", "pres_itemnos", "recalls"):
    print(name, data[name].shape, data[name].dtype)
print("Study item codes:", data["pres_itemnos"][0])
print("Global item IDs: ", data["pres_itemids"][0])

subject (492, 1) int32
listLength (492, 1) int32
pres_itemnos (492, 12) int32
recalls (492, 12) int32
Study item codes: [ 1  2  3  4  5  6  7  8  9 10 11 12]
Global item IDs:  [32 66 48 60 25 59 44 23 22 73 16 41]


## What do the numbers identify?

**Study position**, **within-list item identity** and **cross-list item
identity** serve different purposes. A positive recall code `p` identifies
the item at `pres_itemnos[trial, p - 1]`. The same position in `pres_itemids`
identifies the stimulus across lists. Python indexing starts at zero even
though these stored codes start at one.

In [2]:
#| code-summary: "Translate recall positions into within-list and global item IDs"

positions = data["recalls"][0]
positions = positions[positions > 0]
print("Recall positions:", positions)
print("Model item codes:", data["pres_itemnos"][0, positions - 1])
print("Global item IDs: ", data["pres_itemids"][0, positions - 1])

Recall positions: [ 2  8  5  6 12  9]
Model item codes: [ 2  8  5  6 12  9]
Global item IDs:  [66 23 25 59 41 22]


In this list, within-list codes happen to equal study positions. For repeated
presentations such as `[1, 2, 1]`, positions 1 and 3 identify the same item.
A recorded recall of position 3 translates to model item 1. Dataset simulation
translates that item back to its **first occurrence**, position 1;
repetition-aware analyses can account for both presentations.

Within-list codes must fit the chosen model's item capacity. Ordinary CMR's
factory allocates `max(listLength)` item slots. Global photograph IDs belong
in `pres_itemids`, not directly in `recalls` or the model's item slots.

Zero pads unused study and response slots. Inside a model, `experience(0)`
ignores padding, while `retrieve(0)` stops retrieval. A loss decides which
events are scored; padding alone does not establish an observed stopping
event. Raw responses, intrusions and repeated recalls require an explicit
scoring policy before constructing these arrays.

## What additional information belongs here?

| Alignment | Examples | What must stay aligned |
|---|---|---|
| Trial | `session`, `list_type`, `list_number` | Rows across every field. |
| Study event | `pres_itemids`, `valence` | Rows and study positions. |
| Recall event | `rec_itemids`, `irt` | Rows and response positions. |

Additional fields retain the trial axis even when they contain several
measurements per event. Vocabulary-wide features, distance matrices and
descriptive metadata stay outside the dictionary. Response-dependent fields
need replacement or omission when simulating new responses; see
[Data Simulation](simulation.ipynb).

## What must a particular operation receive?

Semantic procedures need global item IDs and appropriately aligned features
or distances. The supplied distance matrix uses global ID minus one as its
row and column index. Emotional model factories may require study-aligned
`valence`. These requirements are additional to the four core fields.

Padding and variable lengths also matter: some analyses use a fixed list
width in their denominators. The mixed lists used in the guide are all
complete twelve-item lists. Elsewhere, select suitable trials or use an
analysis that accounts for valid study slots. In this dataset, neutral
valence is coded zero: `valence != 0` is therefore not a valid-item mask.

## How is this related to other formats?

The array conventions and familiar field names follow the MATLAB recall
analysis tradition represented by [EMBAM](https://github.com/vucml/EMBAM).
This background does not require a MATLAB file or HDF5.

[Psifr's input representation](https://psifr.readthedocs.io/en/stable/guide/import.html)
is an event table: one row per study or recall event, with participant, list,
phase, position and item fields. JAXCMR's dataset procedures receive aligned,
scored arrays. Converting requires decisions about coding, repetitions and
padding, not just a different file extension.

Storage and representation are separate choices. CSV or HDF5 can contain
experimental information, but the object passed to a JAXCMR function must
match its interface. The library does not provide a general CSV/Psifr
converter.

## How do I construct, load and save it?

`make_dataset()` creates the four core fields for a simple fixed-length
case. It accepts a one-dimensional recall vector as one trial. Supply the
list length explicitly rather than deriving study length from how many
recalls occurred. Optional measurements and raw-response scoring remain
the caller's responsibility.

In [3]:
#| code-summary: "Construct one trial and round-trip its arrays through HDF5"

from pathlib import Path
from tempfile import TemporaryDirectory
import h5py
import jax.numpy as jnp
from jaxcmr.helpers import make_dataset, save_dict_to_hdf5

one_trial = make_dataset(
    recalls=data["recalls"][0], pres_itemnos=data["pres_itemnos"][0],
    listLength=12, subject=int(data["subject"][0, 0]),
)
with TemporaryDirectory() as folder:
    path = str(Path(folder) / "example.h5")
    save_dict_to_hdf5(one_trial, path)
    with h5py.File(path, "r") as stored:
        print("Stored recalls:", stored["data/recalls"].shape)
    reloaded = load_data(path)
    print("Loaded recalls:", reloaded["recalls"].shape)
    print("Arrays preserved:", all(
        bool(jnp.array_equal(one_trial[key], reloaded[key])) for key in one_trial
    ))

Stored recalls: (12, 1)
Loaded recalls: (1, 12)


Arrays preserved: True


`save_dict_to_hdf5()` writes arrays under `/data`, applying `.T` to each;
`load_data()` applies the same transpose on reading and returns JAX arrays.
This is a convention of this helper pair, not of HDF5 itself. Files with
another layout need their own loading step. The temporary illustration
above leaves the observation file unchanged.

[Recall Analyses](analyses.ipynb) explains how trial selection and participant
aggregation use this representation. The [Templates](../templates.qmd)
provide complete procedures and the shared input files.